# 18 — Testa temperature scaling före/efter
Kör några bilder genom modellen, jämför poängen med och utan temperature scaling, innan vi bestämmer om det ska in i appen permanent.

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from PIL import Image
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=178, margin=40, post_process=False)
epic_model = tf.keras.models.load_model('models/epic_detector_3class_2.keras')

print('Modell laddad.')

## Helper-funktioner
`weighted_epic_score` kopierad exakt från `app.py` — annars jämför vi mot fel formel.

In [ ]:
def prepare_image(image):
    image = image.convert('RGB')
    face = mtcnn(image)
    if face is not None:
        arr = face.permute(1, 2, 0).numpy().astype(np.uint8)
    else:
        arr = np.array(image.resize((178, 178)))
    return np.expand_dims(arr, axis=0)


def weighted_epic_score(p_epic, p_medium, p_thin):
    p_max = max(p_epic, p_medium, p_thin)

    if p_max < 0.40:
        return 40.0

    if p_thin == p_max:
        score = p_thin * 8 + p_epic * 100 + p_medium * 50
        if p_max < 0.7:
            score *= 0.92
    else:
        epic_medium_sum = p_epic + p_medium
        effective_anchor = (
            (p_epic * 100 + p_medium * 65) / epic_medium_sum
            if epic_medium_sum > 0 else 100
        )
        score = (p_epic ** 2) * 100 + p_medium * 65 - p_thin * effective_anchor * 0.5
        if p_thin > 0.05:
            score *= 0.92

    score = float(np.clip(score, 0, 100))
    score = 100 * (score / 100) ** 0.85
    return float(np.clip(score, 0, 100))


def apply_temperature(probs, T):
    """Mjukar upp en sannolikhetsfördelning. T=1 ger samma resultat som original."""
    log_probs = np.log(np.array(probs) + 1e-12)
    scaled = log_probs / T
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / np.sum(exp_scaled)

## Kör några bilder, jämför T=1 (original) mot ett högre T
Byt `TEST_IMAGES` till de specifika bilder du vill testa, och `T` till temperaturen du vill prova.

In [ ]:
T = 2.0  # prova 1.5, 2, 3 — högre = mjukare

TEST_IMAGES = sorted(
    glob.glob('data/reference_test_images/*.jpg') +
    glob.glob('data/reference_test_images/*.jpeg') +
    glob.glob('data/reference_test_images/*.png')
)
print(f'{len(TEST_IMAGES)} bilder hittade.')

for path in TEST_IMAGES:
    img = Image.open(path)
    arr = prepare_image(img)
    preds = epic_model.predict(arr, verbose=0)[0]

    original_score = weighted_epic_score(*preds)

    scaled_preds = apply_temperature(preds, T)
    scaled_score = weighted_epic_score(*scaled_preds)

    print(f'{os.path.basename(path)}')
    print(f'  Original : epic={preds[0]:.3f} medium={preds[1]:.3f} thin={preds[2]:.3f} -> {original_score:.1f}')
    print(f'  T={T}     : epic={scaled_preds[0]:.3f} medium={scaled_preds[1]:.3f} thin={scaled_preds[2]:.3f} -> {scaled_score:.1f}')
    print()

## Grad-CAM på 160040.png
Varför tappade den nya modellen säkerhet på en stor, vit mustasch? Samma Grad-CAM-kod som notebook 11 (manuell lager-för-lager forward pass, robust mot Sequential-modeller).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm


def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError('Inget Conv2D-lager hittat.')


def grad_cam(model, img_array, class_idx, layer_name):
    """Manuell lager-för-lager forward pass — robust mot Sequential-modeller
    med nästlade data-augmentation-lager."""
    x = img_array
    for layer in model.layers:
        x = layer(x, training=False)
        if layer.name == layer_name:
            break

    with tf.GradientTape() as tape:
        x = img_array
        conv_output = None
        for layer in model.layers:
            x = layer(x, training=False)
            if layer.name == layer_name:
                conv_output = x
                tape.watch(conv_output)
        predictions = x
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img, heatmap, alpha=0.4):
    heatmap_resized = np.array(Image.fromarray(np.uint8(255 * heatmap)).resize(img.size))
    colored = cm.jet(heatmap_resized)[:, :, :3]
    colored = np.uint8(255 * colored)
    overlay = np.uint8(np.array(img) * (1 - alpha) + colored * alpha)
    return Image.fromarray(overlay)


last_conv = find_last_conv_layer(epic_model)
print('Sista Conv2D-lagret:', last_conv)

PROBLEM_IMAGE_PATH = 'data/reference_test_images/160040.png'
CLASS_NAMES = ['epic', 'medium', 'thin']

original = Image.open(PROBLEM_IMAGE_PATH)
img_array = prepare_image(original)

preds = epic_model.predict(img_array, verbose=0)[0]
print(f'epic={preds[0]:.3f} medium={preds[1]:.3f} thin={preds[2]:.3f}')

img_for_display = Image.fromarray(img_array[0].astype('uint8'))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_for_display)
axes[0].set_title('Croppat ansikte')
axes[0].axis('off')

for ax, cls_idx, cls_name in zip(axes[1:], [0, 1], ['epic', 'medium']):
    heatmap = grad_cam(epic_model, img_array.astype('float32'), cls_idx, last_conv)
    overlay = overlay_heatmap(img_for_display, heatmap)
    ax.imshow(overlay)
    ax.set_title(f'Grad-CAM ({cls_name}={preds[cls_idx]:.2f})')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Top-end-komprimering på den GAMLA modellen (epic_detector_3class_2.keras)
Rör bara poäng ≥90 — mappar 90-100 till 80-100 — lämnar alla lägre/osäkra poäng helt orörda. Laddar den gamla modellen explicit, oberoende av vad `epic_model` råkar peka på just nu.

In [ ]:
old_model = tf.keras.models.load_model('models/epic_detector_3class_2.keras')
print('Gamla modellen (epic_detector_3class_2.keras) laddad explicit.')


def compress_top_end(score, floor=90.0, new_floor=80.0, gamma=2.5):
    """Mappar [floor, 100] till [new_floor, 100] med en potenskurva.
    gamma>1 gör att ENDAST score==100 (exakt 1.0/0.0/0.0) hamnar på 100 —
    allt annat i intervallet pressas tydligt nedåt, ju högre gamma desto mjukare."""
    if score < floor:
        return score
    frac = (score - floor) / (100.0 - floor)
    curved = frac ** gamma
    return new_floor + (100.0 - new_floor) * curved


TEST_IMAGES_OLD = sorted(
    glob.glob('data/reference_test_images/*.jpg') +
    glob.glob('data/reference_test_images/*.jpeg') +
    glob.glob('data/reference_test_images/*.png')
)
print(f'{len(TEST_IMAGES_OLD)} bilder hittade.\n')

for path in TEST_IMAGES_OLD:
    img = Image.open(path)
    arr = prepare_image(img)
    preds = old_model.predict(arr, verbose=0)[0]

    original_score = weighted_epic_score(*preds)
    compressed_score = compress_top_end(original_score)

    marker = '  <-- komprimerad' if compressed_score != original_score else ''
    print(f'{os.path.basename(path)}: epic={preds[0]:.3f} medium={preds[1]:.3f} thin={preds[2]:.3f}')
    print(f'  Original    -> {original_score:.1f}')
    print(f'  Komprimerad -> {compressed_score:.1f}{marker}')
    print()